In [18]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import dataclasses
from dataclasses import dataclass, field
from typing import List
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from template_modules import EncoderStaticBase, EncoderStaticBaseConfig
from collections.abc import Iterable
from basic_conv1d import bn_drop_lin
import warnings
warnings.filterwarnings('ignore')
from clinical_ts.template_modules import ShapeConfig

In [ ]:
# ----------------------------------------
# Config (matches YAML)
# ----------------------------------------
BATCH_SIZE = 32
EPOCHS     = 40
LR         = 0.001
WEIGHT_DECAY = 0.001
DROPOUT    = 0.5
LIN_FTRS   = [128, 128, 128]

In [ ]:
# ----------------------------------------
# 1. Load data (same as multimodal.py)
# ----------------------------------------
print("Loading data...")
df = pd.read_csv('/fs/dss/home/gaad2403/MDS-ED/src/data/memmap/mds_ed.csv',
                 low_memory=False)
print(f"shape: {df.shape}")

In [ ]:
input_cols = [
    c for c in df.columns
    if c.split("_")[0] in ['biometrics', 'demographics', 'labvalues', 'vitals']
]

# =========================================================
# MEDIAN IMPUTATION (training-based)
# =========================================================
df_train = df[df['general_strat_fold'] < 18]

train_medians = df_train[input_cols].median().to_dict()

for c in input_cols:
    if df[c].isna().sum() > 0:
        df[c] = df[c].fillna(train_medians[c])

df = df.copy()

# =========================================================
# CATEGORICAL / CONTINUOUS SPLIT (SAFE VERSION)
# =========================================================
unique_counts = {c: df[c].nunique(dropna=True) for c in input_cols}

cat_features = [
    c for c in input_cols
    if unique_counts[c] < 10 and df[c].dtype != "float"
]

cont_features = [c for c in input_cols if c not in cat_features]

print(
    f"categorical features: {len(cat_features)} | "
    f"continuous features: {len(cont_features)}"
)

# =========================================================
# VITALS ACUITY SAFETY
# =========================================================
if "vitals_acuity" in df.columns:
    df["vitals_acuity"] = df["vitals_acuity"].fillna(0).astype(int)
    df["vitals_acuity"] = df["vitals_acuity"] - 1

# =========================================================
# ETHNICITY FIX (IMPORTANT CORRECTION)
# =========================================================
eth_cols = [
    'demographics_ethnicity_asian',
    'demographics_ethnicity_black/african',
    'demographics_ethnicity_hispanic/latino',
    'demographics_ethnicity_other',
    'demographics_ethnicity_white'
]

# ONLY create ethnicity column if one-hot exists
if all(c in df.columns for c in eth_cols):
    df["demographics_ethnicity"] = df[eth_cols].values.argmax(axis=1)
    df.drop(columns=eth_cols, inplace=True)

# =========================================================
# UPDATE INPUT COLUMNS (SAFE)
# =========================================================
input_cols = [
    c for c in df.columns
    if c.split("_")[0] in ['biometrics', 'demographics', 'labvalues', 'vitals']
]

cat_features = [c for c in input_cols if c in cat_features]
cont_features = [c for c in input_cols if c not in cat_features]

# =========================================================
# TARGETS (24H MORTALITY MULTI-TASK)
# =========================================================
target_cols = [
    "deterioration_mortality_1d",
    "deterioration_icu_24h",
    "deterioration_vasopressors"
]

target_cols = [c for c in target_cols if c in df.columns]

print("Using targets:", target_cols)

for c in target_cols:
    df[c] = df[c].replace(-999., np.nan)

In [ ]:
# ----------------------------------------
# 3. Train/Val/Test split
# ----------------------------------------
train_df = df[df['general_strat_fold'].isin(range(0, 18))].reset_index(drop=True)
val_df   = df[df['general_strat_fold'] == 18].reset_index(drop=True)
test_df  = df[df['general_strat_fold'] == 19].reset_index(drop=True)

# only first ECG per stay for val/test
val_df   = val_df[val_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)
test_df  = test_df[test_df['general_ecg_no_within_stay'] == 0].reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")